# SpaceX Launch Sites Location Analysis

This notebook replicates the Skills Network Labs module covering SpaceX launch site mapping, success visualization, and proximity analysis using folium, pandas, and interactive plugins.

**Objectives:**
- Map launch sites with coordinates
- Visualize success/failure launches
- Calculate and plot proximity to coastlines, highways, railways, and cities
- Answer interpretive questions on siting strategy


In [ ]:
# System setup
!pip install folium pandas


In [ ]:
import folium
import pandas as pd


In [ ]:
# Download and read SpaceX launch geo dataset
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
spacex_df = pd.read_csv(url)
spacex_df.head()


In [ ]:
# Select and group unique launch sites
launch_sites_df = spacex_df[['Launch Site', 'Lat', 'Long']].groupby('Launch Site', as_index=False).first()
launch_sites_df


In [ ]:
# Initialize folium map centered on NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
site_map


In [ ]:
# Add circles/labels for each launch site
from folium.features import DivIcon
for idx, row in launch_sites_df.iterrows():
    coord = [row['Lat'], row['Long']]
    label = row['Launch Site']

    folium.Circle(coord, radius=1000, color='#007849', fill=True).add_child(folium.Popup(label)).add_to(site_map)
    folium.Marker(coord, icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0), html=f'<div style="font-size: 12; color:#007849;"><b>{label}</b></div>')).add_to(site_map)
site_map


In [ ]:
# Add clustered markers for success/failure
from folium.plugins import MarkerCluster
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else 'red')
marker_cluster = MarkerCluster().add_to(site_map)
for idx, row in spacex_df.iterrows():
    folium.Marker([row['Lat'], row['Long']], icon=folium.Icon(color=row['marker_color']), popup=f"{row['Launch Site']}: {'Success' if row['class']==1 else 'Fail'}").add_to(marker_cluster)
site_map


## Task 3: Calculate Distance to Proximities

Use the MousePosition plugin to identify the coordinates for nearby coastlines, railways, highways, and cities. Calculate and map the distance between launch site and each proximity feature.

In [ ]:
from folium.plugins import MousePosition
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(position='topright', separator=' Long: ', empty_string='NaN', lng_first=False, num_digits=20, prefix='Lat:', lat_formatter=formatter, lng_formatter=formatter)
site_map.add_child(mouse_position)
site_map


In [ ]:
# Distance calculation function
from math import radians, cos, sin, asin, sqrt
def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2*asin(sqrt(a))
    km = 6371 * c
    return km


In [ ]:
# Example proximity: coastline
coastline_lat = 28.56367
coastline_lon = -80.57163
launch_site_lat = 28.573255
launch_site_lon = -80.646895
distance_coastline = haversine(launch_site_lon, launch_site_lat, coastline_lon, coastline_lat)
folium.PolyLine([[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]], color='blue', weight=2.5, popup=f"Distance: {distance_coastline:.2f} KM").add_to(site_map)
folium.Marker([coastline_lat, coastline_lon], icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0), html=f'<div style="font-size: 12; color:#d35400;"><b>{distance_coastline:.2f} KM</b></div>')).add_to(site_map)
site_map


In [ ]:
# Example proximity: highway
highway_lat = 28.5660
highway_lon = -80.5800
distance_highway = haversine(launch_site_lon, launch_site_lat, highway_lon, highway_lat)
folium.PolyLine([[launch_site_lat, launch_site_lon], [highway_lat, highway_lon]], color='purple', weight=2.5, popup=f"Distance: {distance_highway:.2f} KM").add_to(site_map)
folium.Marker([highway_lat, highway_lon], icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0), html=f'<div style="font-size: 12; color:#a020f0;"><b>{distance_highway:.2f} KM</b></div>')).add_to(site_map)
site_map


## Findings & Interpretation

- Launch sites are close to coastlines (often within 1 km) for safer trajectories over water.
- Railroads/highways are nearby, supporting logistics and equipment transport.
- Sites are purposefully distant from large cities, maximizing safety while enabling access.

**Checklist answers:**
- Railways: Yes, all NASA/SpaceX sites are adjacent to spur or service tracks.
- Highways: Yes, all sites connect by major road for logistics.
- Coastline: Yes, all launch sites are coastal.
- Cities: None are in city centers; buffer distances maintained.
